In [2]:
!pip install ultralytics roboflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 87.6 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/yolov8_helmet"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {SAVE_DIR}")

Mounted at /content/drive
Checkpoints will be saved to: /content/drive/MyDrive/yolov8_helmet


In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="ROBOFLOW_API_KEY")  # 🔑 replace this
project = rf.workspace("mmoiz-17l7j").project("rider-8o9gz")
dataset = project.version(1).download("yolov8")

print("Dataset location:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Rider-1 in yolov8:: 100%|██████████| 1893/1893 [00:00<00:00, 5946.23it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset location: /content/Rider-1


In [5]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="helmet_LP",
    project=SAVE_DIR,       # saves everything to your Drive folder
    save=True,              # save checkpoints
    save_period=10,         # save a checkpoint every 10 epochs
    device=0,               # GPU
    patience=50,            # early stopping if no improvement for 20 epochs
    exist_ok=True,
)

print("✅ Training done. Weights saved to:", f"{SAVE_DIR}/helmet_LP/weights/")


Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Rider-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=helmet_LP, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=50, perspective=0.

In [7]:
best_model = YOLO(f"{SAVE_DIR}/helmet_LP/weights/best.pt")
metrics = best_model.val()

print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")


Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1802.7±787.2 MB/s, size: 64.4 KB)
val: Scanning /content/Rider-1/valid/labels.cache... 142 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 142/142 59.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 2.3it/s 3.9s
                   all        142       1064      0.753      0.756      0.773      0.354
                helmet        126        327      0.715      0.844      0.807      0.367
              nohelmet         59        179      0.705      0.587      0.659      0.264
           numberplate        117        258      0.802      0.692      0.712      0.265
                 rider        125        300       0.79        0.9      0.913      0.519
Speed: 3.9ms preprocess, 9.8ms inference, 0.0ms l

In [ ]:
# Uncomment and run this cell ONLY if your Colab session crashes mid-training

# from ultralytics import YOLO
# model = YOLO(f"{SAVE_DIR}/rider_detector/weights/last.pt")  # last.pt = latest checkpoint
# model.train(resume=True)

In [ ]:
results = best_model.predict(
    source="YOUR_IMAGE_PATH",   # path, URL, or folder
    conf=0.4,
    save=True,
    project=SAVE_DIR,
    name="inference_results"
)
results[0].show()